# Cassini Dust Analyzer: Interactive Spectra Viewer & Inference Data Filter

This notebook demonstrates:
1. Downloading the level-2 inference dataset (`cda_qm_spectra_pre2008277_inf_lvl2.parquet`) from Hugging Face.
2. Filtering the inference data based on target charge (`qi_ampl` / QI column) between **10 fC** ($10^{-14}$ C) and **2 pC** ($2 \times 10^{-12}$ C).
3. Building an interactive `ipywidgets` tool to load training spectra, select a dust class via a dropdown menu, and click through up to 20 log10-scaled spectra per class.

In [7]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import huggingface_hub
import ipywidgets as widgets
from ipywidgets import interact

print("Setup complete.")

Setup complete.


In [ ]:
# 1. Download and load inference dataset (lvl2)
REPO_ID = "CosmicDustGroup/cassini-cda-spectra"
FILENAME_INF = "data/lvl2/cda_qm_spectra_pre2008277_inf_lvl2.parquet"

print("Downloading inference dataset from Hugging Face...")
file_inf_path = huggingface_hub.hf_hub_download(repo_id=REPO_ID, filename=FILENAME_INF, repo_type="dataset")
print(f"Loaded path: {file_inf_path}")

inf_df = pd.read_parquet(file_inf_path)
print(f"Inference dataset loaded. Shape: {inf_df.shape}")

In [ ]:
# 2. Filter inference data on QI column (qi_ampl) between 10 fC and 2 pC
# 10 fC = 1e-14 C
# 2 pC  = 2e-12 C
QI_MIN = 1e-14
QI_MAX = 2e-12

filtered_inf_df = inf_df[(inf_df['qi_ampl'] >= QI_MIN) & (inf_df['qi_ampl'] <= QI_MAX)].copy()
print(f"Filtered inference dataset shape: {filtered_inf_df.shape} (out of {len(inf_df)} total)")
print("\nFiltered QI Amplitude Statistics (qi_ampl):")
print(filtered_inf_df['qi_ampl'].describe())

In [ ]:
# 3. Download and load training dataset (lvl2) for class-labeled interactive widget
FILENAME_TRAIN = "data/lvl2/cda_qm_spectra_pre2008277_train_lvl2.parquet"

print("Downloading training dataset for class dropdown...")
file_train_path = huggingface_hub.hf_hub_download(repo_id=REPO_ID, filename=FILENAME_TRAIN, repo_type="dataset")
train_df = pd.read_parquet(file_train_path)
print(f"Training dataset loaded. Shape: {train_df.shape}")

# Preprocess spectrum lengths and crop (channels 10-640)
train_df = train_df[train_df['spectrum'].apply(len) == 1018].copy()
train_df['spectrum'] = train_df['spectrum'].apply(lambda s: s[10:641])

# Map labels to academic classes
def map_label(x):
    if not isinstance(x, str) or x in ["?", "X", "2-X"]:
        return "?"
    if x == "3-P":
        return "3-P"
    if x.startswith("3-") or x == "3":
        return "3"
    return x

train_df['class'] = train_df['class'].apply(map_label)
train_df = train_df[train_df['class'] != '?'].copy()
print(f"Preprocessed training dataset shape: {train_df.shape}")

Training dataset loaded. Shape: (18883, 7)
Preprocessed training dataset shape: (18325, 7)


In [ ]:
copy_df = train_df[(train_df['qi_ampl'] >= QI_MIN) & (train_df['qi_ampl'] <= QI_MAX)].copy()

In [12]:
copy_df.loc[copy_df["class"]=="3-P"]

,sclk,f_desc,event_id,spectrum,class,qi_ampl,spectrum_length
440,1509344418,6405611,33776,"[1.7465848e-07, 3.4931696e-07, 3.4931696e-07, ...",3-P,2.080320e-14,1018
1337,1514178834,6405611,18561,"[0.0, 5.239755e-07, 6.986339e-07, -1.7465848e-...",3-P,1.602680e-14,1018
2444,1489096588,6405611,62056,"[0.0, 5.239755e-07, 0.0, 5.239755e-07, 5.23975...",3-P,2.516260e-14,1018
3249,1506224477,6405611,19277,"[6.986339e-07, 5.239755e-07, 3.4931696e-07, 0....",3-P,2.443070e-13,1018
3911,1543799120,6405611,22218,"[0.0, 3.4931696e-07, 3.4931696e-07, 1.0742443e...",3-P,6.385830e-14,1018
3922,1543799233,6405611,22331,"[8.325037e-06, 2.4452188e-06, 2.2705603e-06, 9...",3-P,2.546280e-13,1018
9415,1519607252,6405611,58517,"[0.0, 5.239755e-07, 3.4931696e-07, 1.7465848e-...",3-P,2.786640e-14,1018
11156,1521642257,6405611,408,"[-1.7465848e-07, 3.4931696e-07, 5.239755e-07, ...",3-P,5.913190e-14,1018
17805,1557519144,6405611,30235,"[3.4931696e-07, 5.239755e-07, 5.239755e-07, -3...",3-P,1.858880e-14,1018


In [ ]:
ci

interactive(children=(Dropdown(description='Class:', options=('1', '2', '3', '3-P', '4', '5', '5-Na', 'Noise')…

In [ ]:
# 4. Sample maximum of 20 spectra per class using list comprehension to preserve the grouping column in pandas 2.0+
sampled_df = pd.concat([group.sample(n=min(20, len(group)), random_state=42) for name, group in train_df.groupby('class')]).reset_index(drop=True)

# Helper function to convert raw spectrum into log10 representation with safety offset
def get_log10_spectrum(spec):
    spec = np.array(spec, dtype=float)
    log_spec = np.log10(spec + np.abs(np.min(spec)) + 1e-12)
    finite_mask = np.isfinite(log_spec)
    min_finite_val = np.min(log_spec[finite_mask]) if np.any(finite_mask) else 0
    log_spec = np.nan_to_num(log_spec, neginf=min_finite_val, nan=min_finite_val)
    return log_spec

# Plotting callback for interactive widget
def plot_spectrum(cls, index):
    class_df = sampled_df[sampled_df['class'] == cls]
    num_available = len(class_df)
    if num_available == 0:
        print("No spectra available for this class.")
        return
    
    # Safely bound the index to the actual available count
    idx_clamped = min(index, num_available - 1)
    row = class_df.iloc[idx_clamped]
    log_spec = get_log10_spectrum(row['spectrum'])
    
    plt.figure(figsize=(12, 5))
    plt.plot(log_spec, color='black', linewidth=1.5)
    plt.title(f"Class: {cls} | Spectrum {idx_clamped + 1} of {num_available} (SCLK: {row['sclk']}, QI: {row['qi_ampl']:.2e} C)")
    plt.xlabel("Time-of-Flight Channel Index (cropped 10-640)")
    plt.ylabel("Log10(Amplitude)")
    plt.grid(True)
    plt.ylim(log_spec.min() - 0.05, log_spec.max() + 0.05)
    plt.show()

# Create the interactive widget controls
classes_list = sorted(sampled_df['class'].unique())
class_dropdown = widgets.Dropdown(options=classes_list, value=classes_list[0], description='Class:')
index_slider = widgets.IntSlider(min=0, max=19, step=1, value=0, description='Index:')

# Dynamically update slider max limit based on selected class
def update_slider_range(*args):
    class_df = sampled_df[sampled_df['class'] == class_dropdown.value]
    max_val = max(0, len(class_df) - 1)
    index_slider.max = max_val
    if index_slider.value > max_val:
        index_slider.value = max_val

class_dropdown.observe(update_slider_range, 'value')
update_slider_range()

interact(plot_spectrum, cls=class_dropdown, index=index_slider);